# 1. Initialize Model

In [17]:
#Imports

import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
from torch import nn

from src.models.mlp import MLP
from src.training.train import train, test
from data.dataset import get_dataloader_cifar10, load_cifar10

In [8]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

model = MLP().to(device)

Using mps device


In [9]:
model

MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=3072, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)

# 2. Load data

In [18]:
train_dataset, test_dataset = load_cifar10(normalize=True)

In [10]:
train_dataloader, test_dataloader = get_dataloader_cifar10()

In [11]:
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 3, 32, 32])
Shape of y: torch.Size([64]) torch.int64


# 3. Train Model

In [12]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [23]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.567406  [   64/50000]
loss: 1.542292  [ 6464/50000]
loss: 1.295380  [12864/50000]
loss: 1.694266  [19264/50000]
loss: 1.564343  [25664/50000]
loss: 1.663577  [32064/50000]
loss: 1.626053  [38464/50000]
loss: 1.596990  [44864/50000]
Test Error: 
 Accuracy: 45.2%, Avg loss: 1.574437 

Epoch 2
-------------------------------
loss: 1.552655  [   64/50000]
loss: 1.527858  [ 6464/50000]
loss: 1.277264  [12864/50000]
loss: 1.685813  [19264/50000]
loss: 1.549249  [25664/50000]
loss: 1.646745  [32064/50000]
loss: 1.613021  [38464/50000]
loss: 1.585342  [44864/50000]
Test Error: 
 Accuracy: 45.6%, Avg loss: 1.563104 

Epoch 3
-------------------------------
loss: 1.539176  [   64/50000]
loss: 1.514889  [ 6464/50000]
loss: 1.260704  [12864/50000]
loss: 1.677037  [19264/50000]
loss: 1.535109  [25664/50000]
loss: 1.631486  [32064/50000]
loss: 1.601274  [38464/50000]
loss: 1.574440  [44864/50000]
Test Error: 
 Accuracy: 45.9%, Avg loss: 1.552402 

Epoc

## Test with single image

In [29]:
test_tensor = train_dataset[0][0].to(device)
test_tensor = test_tensor.unsqueeze(0)

In [31]:
model.eval()
pred = model(test_tensor)

In [34]:
target = torch.tensor([train_dataset[0][1]], dtype=torch.long).to(device)
loss = loss_fn(pred, target)

In [35]:
loss

tensor(0.7267, device='mps:0', grad_fn=<NllLossBackward0>)

In [36]:
target

tensor([6], device='mps:0')